# Prep 02 — Image Variant Dataset (rendered HTML screenshots)

**Run this once, before the workshop.**

## What this notebook does

1. Streams `phreshphish/phreshphish` (no full download).
2. For each row, renders the HTML to a PNG screenshot using **Playwright + headless Chromium**.
3. Targets 1k phish + 1k benign successfully-rendered images. Skips failures (timeouts, broken HTML).
4. Pushes to HF as a multimodal dataset under config `image`: each row has `image` (PNG bytes) + `prompt` + `completion`.

## ⚠️ Known limitation: rendering quality is degraded

PhreshPhish stores **HTML only** — not the original rendered DOM. Re-rendering this HTML in 2026 produces visually degraded pages because:

- Many pages are JavaScript-rendered SPAs; with JS disabled (required for safety) they paint as empty shells
- External assets (logos, hero images, CSS, fonts) referenced by the HTML are often offline now
- Big brand auth pages (Facebook, Google, Microsoft) actively serve degraded UIs to non-JS / headless browsers

We mitigate this by:

- **Allowing images / CSS / fonts to load** from the network (was: blocked everything)
- **Disabling the JavaScript engine** at the browser-context level (no XHR, no auto-redirects, no auto-form-submit)
- **Blocking script / websocket / media resource fetches** (saves bandwidth, prevents tracking pings)
- **15s hard timeout** per page; failures are silently skipped

Realistic outcome: ~70–80% of attempts succeed, and of those, maybe 50% look reasonably page-like. The rest are partial / blank-shell renders. **For a workshop demonstrating the SFT mechanic, this is fine** — model is learning a real signal even from imperfect inputs. For production phishing classification you'd want screenshots captured at crawl time (e.g. PhishIRIS-style datasets).

## Where to run

SageMaker Studio JupyterLab. Need ~3 GB extra disk for Chromium. **Don't run on a laptop with <10 GB free.**

## 0. Install dependencies + Chromium

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# Install Chromium for Playwright. ~250MB download.
!playwright install chromium
!playwright install-deps chromium 2>/dev/null || true   # may require sudo on some envs

## 1. Configuration

In [ ]:
HF_REPO_ID = "gt2026workshop/phreshphish-2k"  # text + image variants share this repo via configs
HF_CONFIG_NAME = "image"

TARGET_PER_LABEL = 1000
VIEWPORT = (1024, 768)         # render canvas
PAGE_TIMEOUT_MS = 15_000       # network is in the loop now (was 10_000); leave more headroom
MAX_HTML_BYTES = 1_500_000     # cap input HTML; PhreshPhish has 76MB outliers
MAX_PNG_BYTES = 500_000        # ~500KB per PNG
MAX_CLIP_HEIGHT = 4096         # full-page screenshot height cap
VAL_FRACTION = 0.10
RANDOM_SEED = 42
MAX_EXAMINED = 12_000          # safety bound on streaming iteration (raised: more retries needed)

print(f"Will push to: https://huggingface.co/datasets/{HF_REPO_ID}  (config: {HF_CONFIG_NAME})")

## 2. Render HTML → PNG

Strategy:
- Set HTML directly into the page via `page.set_content(...)` — never let the browser navigate to the live phishing URL (safer, faster, deterministic).
- **Disable JavaScript** at the context level — neutralizes any redirect/auto-submit/exfiltration.
- **Block scripts, websockets, media** at the route level — saves bandwidth, prevents tracking pings.
- **Allow images, CSS, fonts** to fetch from the network — they're the visual signal we want.
- Wait for `load` (not just `domcontentloaded`) + 800ms settle so styles can paint.
- Capture full page **with a 4,096-pixel height cap** — avoids occasional 50,000 px PNGs from runaway content.

In [ ]:
import io
from PIL import Image
from playwright.async_api import async_playwright

# Allow PIL to load tall full-page screenshots (some pages are huge)
Image.MAX_IMAGE_PIXELS = 500_000_000

class HTMLRenderer:
    """Renders raw HTML to PNG bytes with safety + visual-quality tradeoffs.

    Safety: JS engine disabled at context level → no XHR/fetch, no auto-redirect,
    no auto-form-submit. Scripts/websockets/media blocked at the route level.

    Visual quality: images, CSS, fonts ARE allowed to fetch from the network so
    pages have a chance to look like real pages.
    """

    def __init__(self, viewport=(1024, 768), nav_timeout_ms=15_000, max_clip_height=4096):
        self.viewport = viewport
        self.nav_timeout_ms = nav_timeout_ms
        self.max_clip_height = max_clip_height
        self._pw = None
        self._browser = None
        self._context = None

    async def __aenter__(self):
        self._pw = await async_playwright().start()
        self._browser = await self._pw.chromium.launch(headless=True)
        self._context = await self._browser.new_context(
            viewport={"width": self.viewport[0], "height": self.viewport[1]},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 14_0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            java_script_enabled=False,   # ← key safety toggle
        )

        async def _route(route):
            # Block scripts/websockets/media; allow images, css, fonts, stylesheet
            if route.request.resource_type in ("script", "websocket", "media", "eventsource"):
                await route.abort()
            else:
                await route.continue_()
        await self._context.route("**/*", _route)
        return self

    async def __aexit__(self, *exc):
        await self._context.close()
        await self._browser.close()
        await self._pw.stop()

    async def render(self, html: str) -> bytes | None:
        if len(html) > MAX_HTML_BYTES:
            html = html[:MAX_HTML_BYTES]
        page = await self._context.new_page()
        try:
            # wait_until="load" gives CSS/images a chance to paint
            await page.set_content(html, wait_until="load", timeout=self.nav_timeout_ms)
            await page.wait_for_timeout(800)  # extra settle for late-painting assets

            # Cap full-page capture at max_clip_height so we don't produce 50,000px PNGs
            try:
                content_h = await page.evaluate(
                    "() => Math.min(document.documentElement.scrollHeight,"
                    " document.body ? document.body.scrollHeight : 0) || 0"
                )
            except Exception:
                content_h = 0
            clip_h = max(self.viewport[1], min(int(content_h) or self.viewport[1], self.max_clip_height))
            png = await page.screenshot(
                clip={"x": 0, "y": 0, "width": self.viewport[0], "height": clip_h},
                type="png",
            )
            return png
        except Exception:
            return None
        finally:
            await page.close()


def shrink_png(png_bytes: bytes, max_bytes: int = MAX_PNG_BYTES, max_dim: int = 1024) -> bytes:
    """Re-encode PNG to fit under max_bytes. Resize if needed."""
    try:
        img = Image.open(io.BytesIO(png_bytes))
        img.load()
    except Image.DecompressionBombError:
        return png_bytes  # too tall to safely resize; keep raw
    if max(img.size) > max_dim:
        img.thumbnail((max_dim, max_dim))
    buf = io.BytesIO()
    img.save(buf, format="PNG", optimize=True)
    if buf.getbuffer().nbytes > max_bytes:
        rgb = img.convert("RGB")
        for q in (85, 70, 55, 40):
            buf = io.BytesIO()
            rgb.save(buf, format="JPEG", quality=q, optimize=True)
            if buf.getbuffer().nbytes <= max_bytes:
                break
    return buf.getvalue()

## 3. Stream + render loop

In [ ]:
INSTRUCTION = (
    "You are a phishing-detection classifier. Look at the screenshot of a webpage "
    "and output exactly one word: 'phish' if the page is a phishing attempt, "
    "'benign' if it is legitimate."
)
PROMPT = f"{INSTRUCTION}\n\nLabel:"

import asyncio, random
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)

async def collect():
    stream = load_dataset("phreshphish/phreshphish", split="train", streaming=True)
    buckets = {"phish": [], "benign": []}
    skipped = {"empty_html": 0, "render_failed": 0, "already_full": 0}
    examined = 0

    pbar = tqdm(total=2 * TARGET_PER_LABEL, desc="rendered")
    async with HTMLRenderer(viewport=VIEWPORT, nav_timeout_ms=PAGE_TIMEOUT_MS, max_clip_height=MAX_CLIP_HEIGHT) as r:
        for row in stream:
            examined += 1
            if examined > MAX_EXAMINED:
                break

            label = row.get("label")
            if label not in buckets:
                continue
            if len(buckets[label]) >= TARGET_PER_LABEL:
                if all(len(v) >= TARGET_PER_LABEL for v in buckets.values()):
                    break
                continue

            html = row.get("html") or ""
            if not html.strip():
                skipped["empty_html"] += 1
                continue

            png = await r.render(html)
            if png is None:
                skipped["render_failed"] += 1
                continue

            png = shrink_png(png)
            buckets[label].append({
                "image_bytes": png,
                "prompt": PROMPT,
                "completion": label,
                "label": label,
                "target_brand": row.get("target"),
                "sha256": row.get("sha256"),
                "url": row.get("url"),
            })
            pbar.update(1)
            if examined % 100 == 0:
                pbar.set_postfix(examined=examined, **{k: len(v) for k, v in buckets.items()}, **skipped)

    pbar.close()
    return buckets, skipped, examined

buckets, skipped, examined = await collect()
print(f"examined={examined}  phish={len(buckets['phish'])}  benign={len(buckets['benign'])}  skipped={skipped}")

## 4. Quick visual sanity check

In [ ]:
from IPython.display import display
for r in buckets["phish"][:2] + buckets["benign"][:2]:
    print(f"label={r['label']}  brand={r.get('target_brand')}  url={r.get('url','')[:80]}")
    display(Image.open(io.BytesIO(r["image_bytes"])))

## 5. Train/val split + push to HF

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage

all_rows = buckets["phish"] + buckets["benign"]
rng = random.Random(RANDOM_SEED)
rng.shuffle(all_rows)

n_val = int(len(all_rows) * VAL_FRACTION)
val_rows = all_rows[:n_val]
train_rows = all_rows[n_val:]
print(f"train={len(train_rows)}  val={len(val_rows)}")

def to_ds(rows):
    return Dataset.from_dict({
        "image": [{"bytes": r["image_bytes"], "path": None} for r in rows],
        "prompt": [r["prompt"] for r in rows],
        "completion": [r["completion"] for r in rows],
        "label": [r["label"] for r in rows],
        "target_brand": [r.get("target_brand") for r in rows],
        "sha256": [r.get("sha256") for r in rows],
        "url": [r.get("url") for r in rows],
    }).cast_column("image", HFImage())

dsd = DatasetDict(train=to_ds(train_rows), validation=to_ds(val_rows))
dsd

In [ ]:
# Authenticate: huggingface-cli login OR set HF_TOKEN env var
dsd.push_to_hub(HF_REPO_ID, config_name=HF_CONFIG_NAME, private=False)
print(f"Done. Dataset: https://huggingface.co/datasets/{HF_REPO_ID}  (config: {HF_CONFIG_NAME})")